In [1]:
# STEP 10.1 — Load trained Model 1
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import PeftModel

BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
ADAPTER_PATH = "outputs/model1_qlora/final"

compute_dtype = torch.bfloat16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=compute_dtype,
)

model = PeftModel.from_pretrained(
    model,
    ADAPTER_PATH,
)

tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_PATH
)

model.eval()

print("Model 1 loaded successfully.")
print("Tokenizer loaded successfully.")


c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0912 10:26:47.840000 33940 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:22<00:00, 12.94it/s]


Model 1 loaded successfully.
Tokenizer loaded successfully.


In [2]:
# STEP 10.2 — Create inference function

def ask_model(messages):
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    # Move inputs to the model's device
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Get only the newly generated tokens
    input_length = inputs["input_ids"].shape[-1]
    new_tokens = output[0][input_length:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()


In [3]:
# STEP 10.6 — Quick behavior test

tests = [
    "I've had stomach pain since yesterday.",
    "I've been coughing for two weeks.",
    "I've been feeling dizzy lately.",
    "Everything is fine, thank you."
]

for patient_message in tests:
    messages = [
        {
            "role": "user",
            "content": patient_message
        }
    ]

    response = ask_model(messages)

    print("=" * 60)
    print("Patient:", patient_message)
    print("Assistant:", response)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Patient: I've had stomach pain since yesterday.
Assistant: Have you had any nausea or vomiting?
Patient: I've been coughing for two weeks.
Assistant: Have you had any fever or chills?
Patient: I've been feeling dizzy lately.
Assistant: Have you had any blurry vision or light sensitivity?
Patient: Everything is fine, thank you.
Assistant: How are you feeling today?


In [6]:
import torch

print("=" * 90)
print("MODEL 1 — COMPLETE CONVERSATIONAL EVALUATION")
print("=" * 90)

device = next(model.parameters()).device

# ============================================================
# GENERATION FUNCTION
# ============================================================

def generate_response(messages, max_new_tokens=80):

    # Create chat-formatted inputs
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True
    )

    # Move tensors to GPU
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Only decode newly generated tokens
    new_tokens = outputs[0, input_ids.shape[-1]:]

    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    return response


# ============================================================
# TEST FUNCTION
# ============================================================

results = []


def run_test(test_number, name, messages, expected_behavior):

    print("\n" + "=" * 90)
    print(f"TEST {test_number} — {name}")
    print("=" * 90)

    response = generate_response(messages)

    # Print conversation
    for message in messages:
        role = message["role"].upper()
        print(f"{role}: {message['content']}")

    print(f"ASSISTANT: {response}")

    # Basic measurements
    question_count = response.count("?")
    word_count = len(response.split())

    print("\n--- Evaluation ---")
    print(f"Question marks : {question_count}")
    print(f"Word count     : {word_count}")
    print(f"Expected       : {expected_behavior}")

    # Basic one-question check
    if question_count <= 1:
        question_status = "PASS"
    else:
        question_status = "CHECK"

    print(f"Question count : {question_status}")

    results.append({
        "test": test_number,
        "name": name,
        "response": response,
        "question_count": question_count,
        "word_count": word_count
    })

    return response


# ============================================================
# TEST 1
# BASIC CONTEXT TRACKING
# ============================================================

run_test(
    1,
    "Basic Context Tracking",
    [
        {
            "role": "user",
            "content": "I've had a headache for five days."
        },
        {
            "role": "assistant",
            "content": "Where exactly is the headache?"
        },
        {
            "role": "user",
            "content": "Mostly on the right side, around my temple."
        }
    ],
    "Should NOT ask location again."
)


# ============================================================
# TEST 2
# DO NOT REPEAT KNOWN INFORMATION
# ============================================================

run_test(
    2,
    "Known Information — Symptom Character",
    [
        {
            "role": "user",
            "content": "I've had stomach pain since yesterday."
        },
        {
            "role": "assistant",
            "content": "What kind of pain are you having?"
        },
        {
            "role": "user",
            "content": "It's a sharp pain."
        }
    ],
    "Should NOT ask what kind of pain again."
)


# ============================================================
# TEST 3
# MULTIPLE KNOWN FACTS
# ============================================================

run_test(
    3,
    "Multiple Known Facts",
    [
        {
            "role": "user",
            "content": "I've had a headache for five days."
        },
        {
            "role": "assistant",
            "content": "Where exactly is the headache?"
        },
        {
            "role": "user",
            "content": "Mostly on the right side, around my temple."
        },
        {
            "role": "assistant",
            "content": "How would you describe the headache?"
        },
        {
            "role": "user",
            "content": "It's throbbing and pretty severe."
        }
    ],
    "Should avoid duration, location, and character."
)


# ============================================================
# TEST 4
# MULTIPLE FACTS IN ONE PATIENT ANSWER
# ============================================================

run_test(
    4,
    "Multiple Facts In One Answer",
    [
        {
            "role": "user",
            "content": "I've had a cough for two weeks."
        },
        {
            "role": "assistant",
            "content": "Is your cough dry or are you bringing up mucus?"
        },
        {
            "role": "user",
            "content": "I'm bringing up yellow mucus and I've also had a fever for three days."
        }
    ],
    "Should recognize the supplied mucus and fever information."
)


# ============================================================
# TEST 5
# VAGUE ANSWER
# ============================================================

run_test(
    5,
    "Vague Patient Answer",
    [
        {
            "role": "user",
            "content": "I've been having stomach pain."
        },
        {
            "role": "assistant",
            "content": "Where exactly is the pain?"
        },
        {
            "role": "user",
            "content": "I'm not really sure."
        }
    ],
    "Should handle uncertainty without endlessly repeating itself."
)


# ============================================================
# TEST 6
# TOPIC CHANGE
# ============================================================

run_test(
    6,
    "Topic Change",
    [
        {
            "role": "user",
            "content": "I've had a headache for two days."
        },
        {
            "role": "assistant",
            "content": "Where exactly is the headache?"
        },
        {
            "role": "user",
            "content": "Actually, I'm more worried about my chest pain."
        }
    ],
    "Should follow the new chest-pain concern."
)


# ============================================================
# TEST 7
# COLLOQUIAL LANGUAGE
# ============================================================

run_test(
    7,
    "Colloquial / Natural Language",
    [
        {
            "role": "user",
            "content": "Bro my head's been killing me since yesterday, mostly on one side."
        }
    ],
    "Should understand informal wording and ask one useful question."
)


# ============================================================
# TEST 8
# SPELLING ERRORS
# ============================================================

run_test(
    8,
    "Spelling Errors",
    [
        {
            "role": "user",
            "content": "I've had stomack pain for two days and it gets wurse after eating."
        }
    ],
    "Should understand the intended symptom despite spelling errors."
)


# ============================================================
# TEST 9
# RED FLAG
# ============================================================

run_test(
    9,
    "Potential Red Flag",
    [
        {
            "role": "user",
            "content": "I suddenly developed severe chest pain and I'm having trouble breathing."
        }
    ],
    "Should recognize potentially urgent symptoms rather than routine questioning."
)


# ============================================================
# TEST 10
# LONG MULTI-TURN ACCUMULATION
# ============================================================

run_test(
    10,
    "Longer Conversation / Information Accumulation",
    [
        {
            "role": "user",
            "content": "I've had a headache for five days."
        },
        {
            "role": "assistant",
            "content": "Where exactly is the headache?"
        },
        {
            "role": "user",
            "content": "Mostly on the right side, around my temple."
        },
        {
            "role": "assistant",
            "content": "How would you describe the headache?"
        },
        {
            "role": "user",
            "content": "It's throbbing."
        },
        {
            "role": "assistant",
            "content": "How severe is the pain?"
        },
        {
            "role": "user",
            "content": "Pretty severe, around 8 out of 10."
        },
        {
            "role": "assistant",
            "content": "Does anything make the headache better or worse?"
        },
        {
            "role": "user",
            "content": "Bright light makes it worse."
        }
    ],
    "Should ask for NEW relevant information, not restart the interview."
)


# ============================================================
# ADDITIONAL BASIC TESTS
# ============================================================

run_test(
    11,
    "Simple Symptom",
    [
        {
            "role": "user",
            "content": "My back has been hurting."
        }
    ],
    "Should ask one focused follow-up."
)


run_test(
    12,
    "Already Detailed Patient",
    [
        {
            "role": "user",
            "content": "I've had a headache for three days, mostly on the right side. "
                       "It is throbbing, about 7 out of 10, and bright light makes it worse."
        }
    ],
    "Should not ask for information already provided."
)


# ============================================================
# FINAL AUTOMATIC SUMMARY
# ============================================================

print("\n\n" + "=" * 90)
print("FINAL EVALUATION SUMMARY")
print("=" * 90)

total_tests = len(results)

one_question_or_less = sum(
    1 for r in results
    if r["question_count"] <= 1
)

multi_question = sum(
    1 for r in results
    if r["question_count"] > 1
)

avg_words = (
    sum(r["word_count"] for r in results) / total_tests
    if total_tests
    else 0
)

print(f"Total tests                  : {total_tests}")
print(f"≤ 1 question mark            : {one_question_or_less}")
print(f"> 1 question mark            : {multi_question}")
print(f"Average generated words      : {avg_words:.2f}")

print("\n" + "-" * 90)
print("TEST-BY-TEST SUMMARY")
print("-" * 90)

for r in results:
    status = "PASS" if r["question_count"] <= 1 else "CHECK"

    print(
        f"Test {r['test']:02d} | "
        f"{status:5s} | "
        f"Questions: {r['question_count']:2d} | "
        f"Words: {r['word_count']:3d} | "
        f"{r['name']}"
    )

print("\n" + "=" * 90)
print("IMPORTANT")
print("=" * 90)

print(
    "PASS here means the model generated <=1 question mark. "
    "It does NOT automatically mean the question is clinically correct."
)

print(
    "\nFor context tests, manually inspect whether the model "
    "avoided repeating information already given."
)

print(
    "\nFor the red-flag test, manually inspect whether the model "
    "appropriately recognizes urgency rather than continuing routine intake."
)

print("=" * 90)

MODEL 1 — COMPLETE CONVERSATIONAL EVALUATION

TEST 1 — Basic Context Tracking
USER: I've had a headache for five days.
ASSISTANT: Where exactly is the headache?
USER: Mostly on the right side, around my temple.
ASSISTANT: Have you had any fever or chills?

--- Evaluation ---
Question marks : 1
Word count     : 7
Expected       : Should NOT ask location again.
Question count : PASS

TEST 2 — Known Information — Symptom Character
USER: I've had stomach pain since yesterday.
ASSISTANT: What kind of pain are you having?
USER: It's a sharp pain.
ASSISTANT: Have you had any nausea or vomiting?

--- Evaluation ---
Question marks : 1
Word count     : 7
Expected       : Should NOT ask what kind of pain again.
Question count : PASS

TEST 3 — Multiple Known Facts
USER: I've had a headache for five days.
ASSISTANT: Where exactly is the headache?
USER: Mostly on the right side, around my temple.
ASSISTANT: How would you describe the headache?
USER: It's throbbing and pretty severe.
ASSISTANT: Have 